# 04. Recent News Summarization Demo

This notebook demonstrates how the fine-tuned BART-base model can summarize a new article that is not part of CNN/DailyMail.

The demo is intended for presentation only. Because a recent article does not have a human reference summary, this notebook does not compute ROUGE or BERTScore. The output should be judged qualitatively for readability, coverage, and factual consistency.

In [2]:
!pip -q install -U "transformers>=4.41" accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 34.2 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from google.colab import drive
drive.mount('/content/drive')

RUN_DIR = Path('/content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2')
CHECKPOINT_ROOT = RUN_DIR / 'checkpoints'

def find_latest_checkpoint(checkpoint_root):
    checkpoint_dirs = [p for p in checkpoint_root.glob('checkpoint-*') if p.is_dir()]
    if not checkpoint_dirs:
        return checkpoint_root
    return sorted(checkpoint_dirs, key=lambda p: int(p.name.split('-')[-1]))[-1]

MODEL_DIR = find_latest_checkpoint(CHECKPOINT_ROOT)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

print('Loaded model:', MODEL_DIR)
print('Device:', DEVICE)

Mounted at /content/drive


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

Loaded model: /content/drive/MyDrive/ML2Final/ml2_final_bart/bart_base_cnn_dm_train50000_src1024_epochs2/checkpoints/checkpoint-4168
Device: cpu


## Paste a Real News Article

Paste the article title, source, and full article body below. For the presentation slide, use a concise title/source and show only the generated 3-4 sentence summary.

In [1]:
ARTICLE_TITLE = 'Paste article title here'
ARTICLE_SOURCE = 'Paste source here, e.g. AP News / Reuters / BBC / NPR'

ARTICLE_TEXT = """
SpaceX’s regulatory filing last week ahead of its IPO revealed a financially smart marriage between its space-launch services division and its profitable Starlink satellite internet operation.

Its AI business looks much shakier.

Revenue in the company’s AI division has so far mostly come from X, a social-media platform that isn’t exactly a pure-play AI venture.

Acquired by Elon Musk in 2022 and merged into SpaceX along with Musk’s xAI earlier this year, the former Twitter supplied “substantially all” of SpaceX’s AI revenue in 2023 and 2024, the filing said.

Sales in 2024 were $2.62 billion, 11.5% less than the prior year and just over half of what Twitter made in the last full year before its acquisition. Revenue for the division climbed to $3.2 billion last year due mostly to rising subscription sales for X and the AI model Grok. But losses ballooned to $6.35 billion as depreciation charges on AI computing equipment grew.

Of course, SpaceX’s AI aspirations are real, as are its investments in the computing infrastructure to achieve them. Musk’s xAI has built some of the largest AI data centers in the world, and they are just starting to generate substantial sales. SpaceX recently signed a deal with Anthropic to rent access to AI chips at its data centers that should put $15 billion or so into its coffers each of the next three years.

It is hard to say exactly how profitable that deal will be, however. Rapidly growing depreciation is likely to significantly offset revenue in the years ahead. And the company continues to splash out on ever more AI chips and equipment: Capital expenses for the AI division were $7.72 billion in the first quarter alone, compared with $12.7 billion for all of last year.

Perhaps more worryingly, though, SpaceX’s decision to rent out a massive amount of AI computing to Anthropic signals that it can’t find more profitable uses of its own for the equipment. That is despite the company training and selling access to Grok, its own frontier AI model.

Renting out AI computing resources to others does make some financial sense. It is certainly better than leaving the equipment idle. But AI cloud-computing is a crowded arena. SpaceX can compete there but has no real sustainable advantage.

That raises questions about whether the company’s AI business can grow as fast as its valuation implies. SpaceX’s acquisition of xAI valued it at $250 billion, a very large figure for a still relatively small business.

Musk has grand plans to put AI data centers in space and settle humans on Mars. Building up a terrestrial AI business could arguably help pave the way for that.

So far, though, the SpaceX AI play is getting a lot less altitude than its rockets.

This is an edition of the WSJ AI & Business newsletter, a weekly digest to help you make sense of AI’s impact on business with news, insights and data from our global team of technology journalists. If you’re not subscribed, sign up here.
California Gov. Gavin Newsom last week ordered his state to study the impact of AI on the workforce and help people displaced from jobs by the technology. That is no big surprise, considering that California’s Silicon Valley firms including Meta Platforms have been pioneers in laying off people in the belief that AI can replace them. That strikes a contrast with a much more laissez-faire approach from President Trump. Trump delayed signing an executive order that would have asked AI developers to preview advanced models to the government, citing concerns about overregulation.
"""

In [4]:
def summarize_article(article, max_source_length=1024, max_summary_length=128):
    inputs = tokenizer(
        article,
        max_length=max_source_length,
        truncation=True,
        padding=True,
        return_tensors='pt',
    ).to(DEVICE)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            num_beams=4,
            max_length=max_summary_length,
            min_length=30,
            no_repeat_ngram_size=3,
            do_sample=False,
        )
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

article = ARTICLE_TEXT.strip()
if not article or article == 'Paste the full article text here.':
    raise ValueError('Paste a real article into ARTICLE_TEXT first.')

summary = summarize_article(article)

print('Article title:', ARTICLE_TITLE)
print('Source:', ARTICLE_SOURCE)
print('\nGenerated summary:\n')
print(summary)

Article title: Paste article title here
Source: Paste source here, e.g. AP News / Reuters / BBC / NPR

Generated summary:

SpaceX’s regulatory filing last week ahead of its IPO revealed a financially smart marriage between its space-launch services division and its profitable Starlink satellite internet operation .
Its AI business looks much shakier than its rockets .
Revenue in the company's AI division has so far mostly come from X, a social-media platform that isn’t exactly a pure-play AI venture .


## Slide Format

Use this format in the presentation:

| Field | Content |
|---|---|
| Article | Title + source |
| Input | Recent real-world news article pasted into the model |
| Model | Fine-tuned `facebook/bart-base` checkpoint from Notebook 01 |
| Output | Generated summary |
| Note | No ROUGE/BERTScore because the article has no reference summary |
